# Regression analysis - LR Batch Size Combined Variation

## First run - sc4422

### Parameters
wcte_regression.yaml

Overrides: 

data.dataset.transforms=['double_cover'] model.conv_pad_mode='circular' \
tasks.train.data_loaders.train.batch_size=1024 \
tasks.evaluate.data_loaders.test.batch_size=2048 \
tasks.train.data_loaders.validation.batch_size=2048 \
tasks.train.data_loaders.train.pre_transforms=[]


### Data/ Paths

Electron run: \
data.split_path='/vols/hyperk/WCTE/ML_2025/cds_geometry/split_paths/wcte_CDS_pgun_e-_3M_mu-_3M_0to1GeV_fixedFC_fixedmu_6M/e_FC.npz' \
Muon run: \
data.split_path='/vols/hyperk/WCTE/ML_2025/cds_geometry/split_paths/wcte_CDS_pgun_e-_3M_mu-_3M_0to1GeV_fixedFC_fixedmu_6M/e_FC.npz' \
data.dataset.geometry_file='/vols/hyperk/WCTE/ML_2025/cds_geometry/geometries/WCTE_geometry_v1.12.21.npz' \
data.dataset.mpmt_positions_file='/vols/hyperk/WCTE/ML_2025/cds_geometry/mpmt_pos_files/WCTE_mpmt_position_correct_251125.npz' \
data.dataset.h5file='/vols/hyperk/WCTE/ML_2025/cds_geometry/h5_files/wcte_CDS_pgun_e-_3M_mu-_3M_0to1GeV_fixedFC_fixedmu_6M.h5'

### Disclaimer

### Outputs
Run directory: /vols/hyperk/users/sc4422/first_run/outputs/regression/electrons/{regression_type}/$DATE

{regression_type} - position, directions, energy

### WatChMaL github version
workshop_2025-91-g158c799

# Setup

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>div.output_scroll { height: 44em; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sys
import h5py
import awkward as ak
import uproot

In [4]:
matplotlib.rcdefaults()  # Reset to defaults first

# Apply ROOT styling without problematic fonts
matplotlib.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'figure.figsize': (8, 6),
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

print("Matplotlib styling applied successfully")


Matplotlib styling applied successfully


In [5]:
# either add WatChMaL repository directory to PYTHONPATH variable or add it here
sys.path.append('/vols/hyperk/users/sc4422/first_run/WatChMaL')

In [6]:
import watchmal.utils.math as math
from analysis.utils.binning import get_binning
from analysis.utils.plotting import plot_legend
from analysis.regression import (WatChMaLPositionRegression, WatChMaLDirectionRegression, WatChMaLEnergyRegression, CombinedRegressionRun,
                                 FitQun1ParticleFit, plot_histograms, plot_resolution_profile, plot_bias_profile, tabulate_statistics)
from analysis.read import FiTQunOutput

Imported analysis code from WatChMaL repository with git version: workshop_2025-91-g158c799


# Processing h5 data

In [7]:
cds_geo_path = '/vols/hyperk/WCTE/ML_2025/cds_geometry'

In [8]:
data_path_new_geo = cds_geo_path + "/h5_files/wcte_CDS_pgun_e-_10M_mu-_10M_0to1GeV_fixedFC_fixed_mu.h5"
h5_file_new_geo = h5py.File(data_path_new_geo, "r")

In [9]:
idxs_path_new_geo_e = cds_geo_path + "/split_paths/wcte_CDS_pgun_e-_10M_mu-_10M_0to1GeV_fixedFC_fixed_mu/e_FC.npz"
test_idxs_new_geo_e  = np.load(idxs_path_new_geo_e, allow_pickle=True)['test_idxs']

idxs_path_new_geo_mu = cds_geo_path + "/split_paths/wcte_CDS_pgun_e-_10M_mu-_10M_0to1GeV_fixedFC_fixed_mu/mu_FC.npz"
test_idxs_new_geo_mu  = np.load(idxs_path_new_geo_mu, allow_pickle=True)['test_idxs']

# idxs_path_new_geo_emu = cds_geo_path + "/split_paths/wcte_CDS_pgun_e-_10M_mu-_10M_0to1GeV_fixedFC_fixed_mu/emu_ALL.npz"
# test_idxs_new_geo_emu  = np.load(idxs_path_new_geo_emu, allow_pickle=True)['test_idxs']

In [10]:
def get_values(h5file,testidxs):

    h5_angles     = np.array(h5file['angles'])[testidxs].squeeze()
    h5_energies   = np.array(h5file['energies'])[testidxs].squeeze()
    h5_positions  = np.array(h5file['positions'])[testidxs].squeeze()
    h5_labels     = np.array(h5file['labels'])[testidxs].squeeze()
    h5_root_files = np.array(h5file['root_files'])[testidxs].squeeze()
    h5_event_ids  = np.array(h5file['event_ids'])[testidxs].squeeze()
    h5_vetos      = np.array(h5file['veto'])[testidxs].squeeze()
    hitsindex     = np.array(h5file['event_hits_index']).squeeze()
    # h5_charge     = np.array(h5file['hit_charge']).squeeze()
    # charge        = h5_charge[hitsindex[testidxs+1] - h5_charge[hitsindex[testidxs]]]
    # h5_times       = np.array(h5file['hit_time']).squeeze()
    # times         = h5_times[hitsindex[testidxs+1] - h5_times[hitsindex[testidxs]]]
    h5_nhits      = hitsindex[testidxs+1] - hitsindex[testidxs]
    h5_directions = math.direction_from_angles(h5_angles)
    h5_fc = np.array(h5file['fully_contained'])[testidxs].squeeze()

    wcte_radius = 159.63
    wcte_half_height = 141.54

    # some extra values to use in cuts or plotting, calculated from the data read from h5 file
    h5_towall = math.towall(h5_positions, h5_angles, tank_half_height=wcte_half_height, tank_radius=wcte_radius)
    h5_dwall = math.dwall(h5_positions, tank_half_height=wcte_half_height, tank_radius=wcte_radius)
    h5_momentum = math.momentum_from_energy(h5_energies, h5_labels)

    return {
        'angles': h5_angles,
        'energies': h5_energies,
        'positions': h5_positions,
        'labels': h5_labels,
        'root_files': h5_root_files,
        'event_ids': h5_event_ids,
        'vetos': h5_vetos,
        'hitsindex': hitsindex,
        'nhits': h5_nhits,
        'directions': h5_directions,
        'fc': h5_fc,
        'towall':h5_towall,
        'dwall':h5_dwall,
        'momentum':h5_momentum,
        # 'charge': charge,
        # 'times': times
    }

In [11]:
# data_new_geo_emu = get_values(h5_file_new_geo, test_idxs_new_geo_emu)
data_new_geo_mu =  get_values(h5_file_new_geo, test_idxs_new_geo_mu)
data_new_geo_e =  get_values(h5_file_new_geo, test_idxs_new_geo_e)

# Processing fitQun data

# Making cuts and extracting data

In [12]:
font = {'family' : 'DejaVu Sans',
        'weight' : 'normal',
        'size'   : 28}
matplotlib.rc('font', **font)
matplotlib.rcParams['figure.figsize'] = (12, 9)
matplotlib.rcParams["figure.autolayout"] = True

In [13]:
def apply_cuts(data):
    fc_cut = (data['fc'] == True) # truth-based cut to exclude non-fully-contained events
    towall_cut = data['towall'] > 100
    nhits_cut = data['nhits'] > 25
    energies_cut = data['energies'] > 800

    # select the true electron and muon events that pass the cuts
    cuts = ( fc_cut
            & nhits_cut
            & towall_cut
            )
    return cuts

cuts_new_geo_mu = apply_cuts(data_new_geo_mu)
cuts_new_geo_e = apply_cuts(data_new_geo_e)

In [14]:
e_idxs_new_geo = data_new_geo_e['labels']==1
mu_idxs_new_geo = data_new_geo_mu['labels']==2

In [15]:
def get_particle_quantities(data, particle_type, cuts=None):
    """
    Extracts quantities for a given particle type from the data dictionary and
    returns a tuple in the order expected by the notebook unpacking.

    Returns: (labels, momentum, positions, directions, angles,
              dwall, towall, cuts_selected, indices)
    """
    particle_mask = data['labels'] == particle_type

    labels = data['labels'][particle_mask]
    momentum = data['momentum'][particle_mask]
    positions = data['positions'][particle_mask]
    directions = data['directions'][particle_mask]
    angles = data['angles'][particle_mask]
    dwall = data['dwall'][particle_mask]
    towall = data['towall'][particle_mask]
    cuts_selected = cuts[particle_mask] if cuts is not None else None
    indices = cuts[particle_mask]

    return (labels, momentum, positions, directions, angles,
            dwall, towall, cuts_selected, indices)

In [16]:
e_labels, e_mom, e_pos, e_dir, e_ang, e_dwall, e_towall, e_cuts, e_idxs = get_particle_quantities(data_new_geo_e, 1, cuts_new_geo_e)
mu_labels, mu_mom, mu_pos, mu_dir, mu_ang, mu_dwall, mu_towall, mu_cuts, mu_idxs = get_particle_quantities(data_new_geo_mu, 2, cuts_new_geo_mu)

In [17]:
resnet_args = {'linestyle':"-"}
electron_args_new_geo = {'indices':test_idxs_new_geo_e[e_idxs_new_geo], 'selection':e_cuts}
muon_args_new_geo = {'indices':test_idxs_new_geo_mu[mu_idxs_new_geo],'selection':mu_cuts} # still selection for the physical cuts

#muon_args_new_geo = {'indices':test_idxs_new_geo[mu_idxs_new_geo],'selection':mu_cuts} # trying to convert “muon rows inside the test subset” into “global H5 row indices” for the output dir

# Loading ResNet results

In [18]:
outdir_reg = "/vols/hyperk/users/sc4422/first_run/outputs/regression/"

In [19]:
# cuts_new_geo_emu = apply_cuts(data_new_geo_emu)
# e_labels, e_mom, e_pos, e_dir, e_ang, e_dwall, e_towall, e_cuts, e_idxs = get_particle_quantities(data_new_geo_emu, 1, cuts_new_geo_emu)
# mu_labels, mu_mom, mu_pos, mu_dir, mu_ang, mu_dwall, mu_towall, mu_cuts, mu_idxs = get_particle_quantities(data_new_geo_emu, 2, cuts_new_geo_emu)

In [20]:
# # # output_energy = "2026-02-05/18-30-30"
# # # output_pos = "2026-02-05/21-17-54"
# # # output_dir = ""

# # output_pos_mu = 'muons/position/2026-02-12/17-14-12/'
# # output_pos_e = 'electrons/position/2026-02-12/10-36-01/'

# # output_dir_mu = 'muons/directions/2026-02-12/16-11-11/'
# # output_dir_e = 'electrons/directions/2026-02-12/11-27-55/'


# output_pos_mu = 'muons/position/2026-02-17/11-01-24/'
# output_pos_e = 'electrons/position/2026-02-17/11-00-22/'

# output_dir_mu = 'muons/directions/2026-02-17/11-30-33/'
# output_dir_e = 'electrons/directions/2026-02-17/11-29-33/'

# # output_energy_mu = 'muons/energy/2026-02-12/18-14-04/'

# output_energy_mu = 'muons/energy/2026-02-17/00-01-04/'

# # output_energy_e = 'electrons/energy/2026-02-12/14-24-01/'

# output_energy_e = 'electrons/energy/2026-02-17/11-51-58/'

### Energy regression only

# output_0007_256_r50_all = 'electrons/lr/2026-03-06/14-56-15'
# output_0007_256_r50_e   = 'electrons/lr/2026-03-06/14-59-30'
# output_001_512_e        = 'electrons/lr/2026-03-06/22-53-22'

# output_001_512_pos = 'electrons/lr/2026-03-10/16-29-35'
# output_0007_256_pos = 'electrons/lr/2026-03-10/23-11-19'

# output_001_512_r50_all_152 = 'electrons/lr/2026-03-09/20-25-49'
# output_0007_256_r50_all_152 = 'electrons/lr/2026-03-09/20-30-49'


# resnet_e_pos = [
#     WatChMaLPositionRegression(
#         outdir_reg + output_001_512_pos,
#         "ResNet50 Electron Position LR 0.001 BS 512",
#         e_pos, e_dir,
#         **electron_args_new_geo, **resnet_args
#     ),
#     WatChMaLPositionRegression(
#         outdir_reg + output_0007_256_pos,
#         "ResNet50 Electron Position LR 0.0007 BS 256",
#         e_pos, e_dir,
#         **electron_args_new_geo, **resnet_args
#     ),

#     WatChMaLPositionRegression(
#         outdir_reg + output_0007_256_r50_all,
#         "ResNet50 Electron Position LR 0.0007 BS 256 All",
#         e_pos, e_dir,
#         **electron_args_new_geo, **resnet_args
#     ),

#     WatChMaLPositionRegression(
#         outdir_reg + output_001_512_r50_all_152,
#         "ResNet50-152 Electron Position LR 0.001 BS 512 All",
#         e_pos, e_dir,
#         **electron_args_new_geo, **resnet_args
#     ),

#     WatChMaLPositionRegression(
#         outdir_reg + output_0007_256_r50_all_152,
#         "ResNet50-152 Electron Position LR 0.0007 BS 256 All",
#         e_pos, e_dir,
#         **electron_args_new_geo, **resnet_args
#     ),
# ]

# resnet_e_energy = [
#     WatChMaLEnergyRegression(
#         outdir_reg + output_0007_256_r50_all,
#         "ResNet50 Electron Energy LR 0.0007 BS 256 All",
#         e_mom, 1,
#         **electron_args_new_geo, **resnet_args
#     ),
#     WatChMaLEnergyRegression(
#         outdir_reg + output_0007_256_r50_e,
#         "ResNet50 Electron Energy LR 0.0007 BS 256 Electron",
#         e_mom, 1,
#         **electron_args_new_geo, **resnet_args
#     ),
#     WatChMaLEnergyRegression(
#         outdir_reg + output_001_512_e,
#         "ResNet50 Electron Energy LR 0.001 BS 512 Electron",
#         e_mom, 1,
#         **electron_args_new_geo, **resnet_args
#     ),
#     WatChMaLEnergyRegression(
#         outdir_reg + output_001_512_r50_all_152,
#         "ResNet152 Electron Energy LR 0.001 BS 512 All",
#         e_mom, 1,
#         **electron_args_new_geo, **resnet_args
#     ),
#     WatChMaLEnergyRegression(
#         outdir_reg + output_0007_256_r50_all_152,
#         "ResNet152 Electron Energy LR 0.0007 BS 256 All",
#         e_mom, 1,
#         **electron_args_new_geo, **resnet_args
#     ),
# ]

In [21]:
output_all_1 = 'electrons/output_channels/all/2026-03-11/14-24-53'
output_all_2 = 'electrons/output_channels/all/2026-03-11/14-24-56'
output_all_3 = 'electrons/output_channels/all/2026-03-11/14-24-58'
output_all_4 = 'electrons/output_channels/all/2026-03-12/11-36-30'
output_all_5 = 'electrons/output_channels/all/2026-03-12/11-36-50'

output_e_1 = 'electrons/output_channels/energy/2026-03-11/14-24-16'
output_e_2 = 'electrons/output_channels/energy/2026-03-11/14-24-37'
output_e_3 = 'electrons/output_channels/energy/2026-03-16/15-00-10'
output_e_4 = 'electrons/output_channels/energy/2026-03-16/15-00-11'
output_e_5 = 'electrons/output_channels/energy/2026-03-16/15-00-12'




resnet_e_energy = [
    WatChMaLEnergyRegression(
        outdir_reg + output_e_1,
        "ResNet50 Electron Energy Run 1 Electron Channel",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_e_2,
        "ResNet50 Electron Energy Run 2 Electron Channel",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_e_3,
        "ResNet50 Electron Energy Run 3 Electron Channel",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_e_4,
        "ResNet50 Electron Energy Run 4 Electron Channel",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_e_5,
        "ResNet50 Electron Energy Run 5 Electron Channel",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_all_1,
        "ResNet50 Electron Energy Run 1 All Channels",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_all_2,
        "ResNet152 Electron Energy Run 2 All Channels",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_all_3,
        "ResNet152 Electron Energy Run 3 All Channels",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_all_4,
        "ResNet152 Electron Energy Run 3 All Channels",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
    WatChMaLEnergyRegression(
        outdir_reg + output_all_5,
        "ResNet152 Electron Energy Run 3 All Channels",
        e_mom, 1,
        **electron_args_new_geo, **resnet_args
    ),
]

# Plot training progression

In [22]:
# for r in resnet_muon_position:
#     fig, ax1 = r.plot_training_progression(fig_size=(8,6), title=r.run_label, plot_best=True)
#     # fig, ax1 = r.plot_validation_progression(legend=None, fig_size=(8,6))
#     ax1.set_yscale('log')
# for r in resnet_muon_direction:
#     fig, ax1 = r.plot_training_progression(legend=None, fig_size=(8,6), title=r.run_label)
# for r in resnet_muon_energy:
#     fig, ax1 = r.plot_training_progression(fig_size=(8,6), title=r.run_label, plot_best=True)
#     # fig, ax1 = r.plot_validation_progression(legend=None, fig_size=(8,6))
#     ax1.set_yscale('log')

# # for r in resnet_muon_energy:
# #     fig, ax1 = r.plot_training_progression(legend=None, fig_size=(8,6), title=r.run_label)
# # for r in resnet_e_position:
# #     fig, ax1 = r.plot_training_progression(legend=None, fig_size=(8,6), title=r.run_label)
# for r in resnet_e_position:
#     fig, ax1 = r.plot_training_progression(fig_size=(8,6), title=r.run_label, plot_best=True)
#     # fig, ax1 = r.plot_validation_progression(legend=None, fig_size=(8,6))
#     ax1.set_yscale('log')
# for r in resnet_e_direction:
#     fig, ax1 = r.plot_training_progression(legend=None, fig_size=(8,6), title=r.run_label)
# for r in resnet_e_energy:
#     fig, ax1 = r.plot_training_progression(legend=None, fig_size=(8,6), title=r.run_label)

In [23]:
for r in resnet_e_pos:
    fig, ax1 = r.plot_training_progression(fig_size=(8,6), title=r.run_label, plot_best=True)
    # fig, ax1 = r.plot_validation_progression(legend=None, fig_size=(8,6))
    ax1.set_yscale('log')

NameError: name 'resnet_e_pos' is not defined

# Statistics

In [ ]:
position_stats = ['x_residuals', 'y_residuals', 'z_residuals', 'position_3d_errors', 'position_transverse_errors', 'position_longitudinal_errors', 'position_longitudinal_errors']
position_stat_labels = ["x position resolution [cm]", "y position resolution [cm]", "z position resolution [cm]", "3D position resolution [cm]", "Transverse position resolution [cm]", "Longitudinal position resolution [cm]", "Longitudinal position mean bias [cm]"]
position_stat_types = ['resolution', 'resolution', 'resolution', 'resolution', 'resolution', 'resolution', 'mean']

tabulate_statistics(resnet_muon_pos, position_stats, position_stat_labels, position_stat_types)



In [ ]:
tabulate_statistics(resnet_e_pos, position_stats, position_stat_labels, position_stat_types)

In [24]:
#hostname
energy_percentage_errors = lambda r: r.energy_fractional_errors*100
momentum_percentage_errors = lambda r: r.momentum_fractional_errors*100

energy_stats = ["energy_residuals", 'momentum_residuals', "energy_residuals", 'momentum_residuals', energy_percentage_errors, momentum_percentage_errors,energy_percentage_errors, momentum_percentage_errors]
# energy_stats = ["energy_fractional_errors", "energy_residuals", 'momentum_residuals', 'momentum_fractional_errors']
energy_stat_labels = ["energy_residuals", 'momentum_residuals', "energy_residuals", 'momentum_residuals', 'energy_percentage_errors', 'momentum_percentage_errors','energy_percentage_errors', 'momentum_percentage_errors']
energy_stat_types = ['resolution', 'resolution', 'mean','mean', 'resolution', 'resolution','mean','mean',]

# r = resnet_muon_energy[0]
# valid = np.asarray(r.predictions) >= 105.7

tabulate_statistics(resnet_muon_energy, energy_stats, energy_stat_labels, energy_stat_types)

NameError: name 'resnet_muon_energy' is not defined

In [25]:
tabulate_statistics(resnet_e_energy, energy_stats, energy_stat_labels, energy_stat_types)

,ResNet50 Electron Energy Run 1 Electron Channel,ResNet50 Electron Energy Run 2 Electron Channel,ResNet50 Electron Energy Run 3 Electron Channel,ResNet50 Electron Energy Run 4 Electron Channel,ResNet50 Electron Energy Run 5 Electron Channel,ResNet50 Electron Energy Run 1 All Channels,ResNet152 Electron Energy Run 2 All Channels,ResNet152 Electron Energy Run 3 All Channels,ResNet152 Electron Energy Run 3 All Channels,ResNet152 Electron Energy Run 3 All Channels
energy_residuals,21.60,22.15,22.57,22.62,22.64,21.32,22.83,21.39,21.13,21.21
momentum_residuals,21.60,22.15,22.57,22.62,22.64,21.32,22.83,21.39,21.13,21.21
energy_residuals,-20.49,-20.76,-16.53,-18.73,-17.22,-22.29,-15.54,-20.25,-20.25,-20.57
momentum_residuals,-20.49,-20.76,-16.53,-18.73,-17.22,-22.29,-15.54,-20.25,-20.25,-20.57
energy_percentage_errors,8.73,8.65,9.00,8.73,9.23,8.35,8.98,8.42,8.30,8.39
momentum_percentage_errors,8.73,8.65,9.00,8.73,9.23,8.35,8.98,8.42,8.30,8.39
energy_percentage_errors,-3.20,-2.68,-0.71,-2.85,-0.95,-3.21,-0.52,-3.67,-2.13,-2.50
momentum_percentage_errors,-3.20,-2.68,-0.71,-2.85,-0.95,-3.21,-0.52,-3.67,-2.12,-2.50


In [ ]:
# Direction statistics

dir_stats = ["direction_errors"]
dir_stat_labels = ["Direction residual error"]
dir_stat_types = ['resolution']
tabulate_statistics(resnet_muon_dir, dir_stats, dir_stat_labels, dir_stat_types)

In [ ]:
tabulate_statistics(resnet_e_dir, dir_stats, dir_stat_labels, dir_stat_types)

# Plots

In [ ]:
# print(len(data_new_geo["positions"][h5_keep_idx][mu_mask_h5]))
# print(len(mu_pos[mu_cuts]))

# A = test_idxs_new_geo[h5_keep_idx][mu_mask_h5]
# B = test_idxs_new_geo[mu_idxs_new_geo][mu_cuts]

# print(len(A), len(B))
# print(np.array_equal(A, B))                 # identical set + order
# print(np.array_equal(np.sort(A), np.sort(B)))  # identical set ignoring order


### Muon Plots

In [ ]:
# from matplotlib.ticker import MaxNLocator, LogLocator
# import matplotlib
# import matplotlib.pyplot as plt

# matplotlib.rcParams.update({
#     'axes.labelsize': 15,
#     'xtick.labelsize': 14,
#     'ytick.labelsize': 14,
# })

# labels = np.asarray(data_new_geo_emu["labels"]).astype(int)

# mu_mask = labels == 2

# fitqun_mu = FitQun1ParticleFit(
#     fq,
#     "fiTQun",
#     true_positions=data_new_geo_emu["positions"][h5_keep_idx][mu_mask],
#     true_directions=data_new_geo_emu["directions"][h5_keep_idx][mu_mask],
#     true_momenta=data_new_geo_emu["momentum"][h5_keep_idx][mu_mask],
#     true_labels=2,                        
#     indices=fq_keep_idx[mu_mask],       
#     selection=cuts_new_geo_mu,
# )

muon_mom_binning = get_binning(data_new_geo_mu["momentum"], 20, 0, 1000)
muon_cos_zenith_binning = get_binning(np.cos(data_new_geo_mu["angles"][:,0]), 20, -1, 1)

muon_azimuth_binning = get_binning(data_new_geo_mu["angles"][:,1] * 180 / np.pi, 20, -180, 180)

muon_dwall_binning = get_binning(data_new_geo_mu["dwall"], 22, 50, 300)

muon_towall_binning = get_binning(data_new_geo_mu["towall"], 30, 50, 800)

In [ ]:
all_runs_pos = resnet_muon_position #+ [fitqun_mu]

figs, axes = plt.subplots(3,2, figsize=(15, 15))
for r in all_runs_pos:
    plot_resolution_profile([r], 'position_3d_errors', muon_towall_binning, ax=axes[0,0], x_label="Distance to detector wall [cm]", y_label="Position resolution [cm]",legend=None)
    plot_resolution_profile([r], 'position_3d_errors', muon_dwall_binning, ax=axes[0,1], x_label="Distance from detector wall [cm]",y_label="Position resolution [cm]",legend=None)
    plot_resolution_profile([r],'position_3d_errors', muon_cos_zenith_binning, ax=axes[1,0], x_label="Cosine of Zenith", y_label="Position resolution [cm]", legend=None)
    plot_resolution_profile([r],'position_3d_errors',muon_azimuth_binning,ax=axes[1,1],x_label="Azimuth",y_label="Position resolution [cm]",legend=None)
    plot_resolution_profile([r],'position_3d_errors',muon_mom_binning, ax=axes[2,0],x_label="True muon momentum [MeV]",y_label="Position resolution [cm]",legend=None)


In [ ]:
all_runs_mom =  resnet_muon_energy #+ [fitqun_mu]

figs, axes = plt.subplots(3,2, figsize=(15, 15))
for r in all_runs_mom:
    plot_resolution_profile([r], 'momentum_residuals', muon_towall_binning, ax=axes[0,0], x_label="Distance to detector wall [cm]", y_label="Momentum resolution",legend=None)
    plot_resolution_profile([r], 'momentum_residuals', muon_dwall_binning, ax=axes[0,1], x_label="Distance from detector wall [cm]",y_label="Momentum resolution",legend=None)
    plot_resolution_profile([r],'momentum_residuals', muon_cos_zenith_binning, ax=axes[1,0], x_label="Cosine of Zenith", y_label="Momentum resolution", legend=None)
    plot_resolution_profile([r],'momentum_residuals',muon_azimuth_binning,ax=axes[1,1],x_label="Azimuth",y_label="Position Momentum",legend=None)
    plot_resolution_profile([r],'momentum_residuals',muon_mom_binning, ax=axes[2,0],x_label="True muon momentum [MeV]",y_label="Momentum resolution",legend=None)


In [ ]:
all_runs_dir =  resnet_muon_direction #+ [fitqun_mu]

figs, axes = plt.subplots(3,2, figsize=(15, 15))
for r in all_runs_dir:
    plot_resolution_profile([r], 'direction_errors', muon_towall_binning, ax=axes[0,0], x_label="Distance to detector wall [cm]", y_label="Direction resolution",legend=None)
    plot_resolution_profile([r], 'direction_errors', muon_dwall_binning, ax=axes[0,1], x_label="Distance from detector wall [cm]",y_label="Direction resolution",legend=None)
    plot_resolution_profile([r],'direction_errors', muon_cos_zenith_binning, ax=axes[1,0], x_label="Cosine of Zenith", y_label="Direction resolution", legend=None)
    plot_resolution_profile([r],'direction_errors',muon_azimuth_binning,ax=axes[1,1],x_label="Azimuth",y_label="Direction Momentum",legend=None)
    plot_resolution_profile([r],'direction_errors',muon_mom_binning, ax=axes[2,0],x_label="True muon momentum [MeV]",y_label="Direction resolution",legend=None)


In [ ]:
# e_mask = labels == 1

# fitqun_e = FitQun1ParticleFit(
#     fq,
#     "fiTQun",
#     true_positions=data_new_geo_emu["positions"][h5_keep_idx][e_mask],
#     true_directions=data_new_geo_emu["directions"][h5_keep_idx][e_mask],
#     true_momenta=data_new_geo_emu["momentum"][h5_keep_idx][e_mask],
#     true_labels=1,                        
#     indices=fq_keep_idx[e_mask],       
#     selection=cuts_new_geo_e,
# )

e_mom_binning = get_binning(data_new_geo_e["momentum"], 20, 0, 1000)

e_cos_zenith_binning = get_binning(np.cos(data_new_geo_e["angles"][:,0]), 20, -1, 1)

e_azimuth_binning = get_binning(data_new_geo_e["angles"][:,1] * 180 / np.pi, 20, -180, 180)

e_dwall_binning = get_binning(data_new_geo_e["dwall"], 22, 50, 300)

e_towall_binning = get_binning(data_new_geo_e["towall"], 30, 50, 800)

all_runs_pos = resnet_e_pos #+ [fitqun_e]

figs, axes = plt.subplots(3,2, figsize=(15, 15))
for r in all_runs_pos:
    plot_resolution_profile([r], 'position_3d_errors', e_towall_binning, ax=axes[0,0], x_label="Distance to detector wall [cm]", y_label="Position resolution [cm]",legend=None)
    plot_resolution_profile([r], 'position_3d_errors',e_dwall_binning, ax=axes[0,1], x_label="Distance from detector wall [cm]",y_label="Position resolution [cm]",legend=None)
    plot_resolution_profile([r],'position_3d_errors', e_cos_zenith_binning, ax=axes[1,0], x_label="Cosine of Zenith", y_label="Position resolution [cm]", legend=None)
    plot_resolution_profile([r],'position_3d_errors', e_azimuth_binning,ax=axes[1,1],x_label="Azimuth",y_label="Position resolution [cm]",legend=None)
    plot_resolution_profile([r],'position_3d_errors',e_mom_binning, ax=axes[2,0],x_label="True electron momentum [MeV]",y_label="Position resolution [cm]",legend=None)

In [ ]:
all_runs_mom =  resnet_e_energy #+ [fitqun_e]

figs, axes = plt.subplots(3,2, figsize=(15, 15))
for r in all_runs_mom:
    plot_resolution_profile([r], 'momentum_residuals', e_towall_binning, ax=axes[0,0], x_label="Distance to detector wall [cm]", y_label="Energy resolution",legend=None)
    plot_resolution_profile([r], 'momentum_residuals', e_dwall_binning, ax=axes[0,1], x_label="Distance from detector wall [cm]",y_label="Energy resolution",legend=None)
    plot_resolution_profile([r],'momentum_residuals', e_cos_zenith_binning, ax=axes[1,0], x_label="Cosine of Zenith", y_label="Energy resolution", legend=None)
    plot_resolution_profile([r],'momentum_residuals',e_azimuth_binning,ax=axes[1,1],x_label="Azimuth",y_label="Energy resolution",legend=None)
    plot_resolution_profile([r],'momentum_residuals', e_mom_binning, ax=axes[2,0],x_label="True electron momentum [MeV]",y_label="Energy resolution",legend=None)

In [ ]:
all_runs_dir =  resnet_e_direction #+ [fitqun_e]

figs, axes = plt.subplots(3,2, figsize=(15, 15))
for r in all_runs_dir:
    plot_resolution_profile([r], 'direction_errors', e_towall_binning, ax=axes[0,0], x_label="Distance to detector wall [cm]", y_label="Direction resolution",legend=None)
    plot_resolution_profile([r], 'direction_errors', e_dwall_binning, ax=axes[0,1], x_label="Distance from detector wall [cm]",y_label="Direction resolution",legend=None)
    plot_resolution_profile([r],'direction_errors', e_cos_zenith_binning, ax=axes[1,0], x_label="Cosine of Zenith", y_label="Direction resolution", legend=None)
    plot_resolution_profile([r],'direction_errors',e_azimuth_binning,ax=axes[1,1],x_label="Azimuth",y_label="Direction Momentum",legend=None)
    plot_resolution_profile([r],'direction_errors', e_mom_binning, ax=axes[2,0],x_label="True electron momentum [MeV]",y_label="Direction resolution",legend=None)

In [ ]:
## seems like the separate oens are worse?

# also plot more condensed and define as a func maybe